# Local Provider

> Local file system discovery provider implementation.

In [ ]:
#| default_exp providers.local

In [ ]:
#| export
import asyncio
import os
from pathlib import Path
from typing import List, Optional

from cjm_file_discovery.core.models import FileInfo, FileType
from cjm_file_discovery.core.config import ScanConfig, ExtensionMapping
from cjm_file_discovery.utils.formatting import get_mime_type

## LocalDiscoveryProvider

Local file system implementation of the DiscoveryProvider protocol.

In [ ]:
#| export
class LocalDiscoveryProvider:
    """Local file system discovery provider."""

    @property
    def name(self) -> str:  # Provider identifier
        """Provider identifier."""
        return "local"

    def supports_path(
        self,
        path: str  # Path to check
    ) -> bool:  # True if this is a local path
        """Check if this provider can handle the given path."""
        # Local provider handles paths that don't have special prefixes
        return not any(path.startswith(prefix) for prefix in ['s3://', 'gs://', 'http://', 'https://'])

    def get_file_info(
        self,
        path: str,  # Path to file
        extension_mapping: Optional[ExtensionMapping] = None  # For type detection
    ) -> Optional[FileInfo]:  # FileInfo or None if not found
        """Get metadata for a single file."""
        try:
            p = Path(path)
            if not p.exists():
                return None

            stat = p.stat()
            ext = p.suffix.lstrip('.').lower() if p.suffix else None
            mapping = extension_mapping or ExtensionMapping()
            file_type = mapping.get_type(ext) if ext else FileType.OTHER

            return FileInfo(
                name=p.name,
                path=str(p.absolute()),
                is_directory=p.is_dir(),
                size=stat.st_size if not p.is_dir() else None,
                modified=stat.st_mtime,
                created=getattr(stat, 'st_birthtime', stat.st_ctime),
                file_type=file_type,
                extension=ext,
                mime_type=get_mime_type(str(p)) if not p.is_dir() else None,
                provider_name=self.name
            )
        except (OSError, PermissionError):
            return None

    def _scan_directory(
        self,
        directory: Path,        # Directory to scan
        config: ScanConfig,     # Scan configuration
        current_depth: int = 0  # Current recursion depth
    ) -> List[FileInfo]:  # List of discovered files
        """Recursively scan a directory."""
        files = []

        try:
            for entry in directory.iterdir():
                # Skip hidden files/directories if not included
                if not config.filter_config.include_hidden and entry.name.startswith('.'):
                    continue

                # Skip symlinks if not following them
                if entry.is_symlink() and not config.follow_symlinks:
                    continue

                if entry.is_dir():
                    # Recurse into subdirectories
                    if config.recursive:
                        # Check depth limit
                        if config.max_depth is not None and current_depth >= config.max_depth:
                            continue
                        files.extend(self._scan_directory(entry, config, current_depth + 1))
                else:
                    # Get file info
                    file_info = self.get_file_info(str(entry), config.extension_mapping)
                    if file_info is not None:
                        # Apply filters
                        if config.filter_config.matches(file_info):
                            files.append(file_info)

        except PermissionError:
            pass  # Skip directories we can't access

        return files

    def scan(
        self,
        directories: List[str],  # Directories to scan
        config: ScanConfig       # Scan configuration
    ) -> List[FileInfo]:  # List of discovered files
        """Scan directories for files."""
        all_files = []

        for dir_path in directories:
            p = Path(dir_path)
            if p.exists() and p.is_dir():
                all_files.extend(self._scan_directory(p, config))

        return all_files

    async def scan_async(
        self,
        directories: List[str],  # Directories to scan
        config: ScanConfig       # Scan configuration
    ) -> List[FileInfo]:  # List of discovered files
        """Async scan for files (runs sync scan in executor)."""
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.scan, directories, config)

In [ ]:
import tempfile
import os

from cjm_file_discovery.core.config import ScanConfig, FilterConfig

# Test LocalDiscoveryProvider
provider = LocalDiscoveryProvider()

# Test supports_path
assert provider.supports_path("/home/user/file.txt") == True
assert provider.supports_path("s3://bucket/file.txt") == False
assert provider.supports_path("https://example.com/file.txt") == False

# Test get_file_info with a real file
with tempfile.NamedTemporaryFile(suffix=".txt", delete=False) as f:
    f.write(b"test content")
    temp_path = f.name

try:
    info = provider.get_file_info(temp_path)
    assert info is not None
    assert info.name.endswith(".txt")
    assert info.is_directory == False
    assert info.size == 12  # "test content" is 12 bytes
    assert info.extension == "txt"
    assert info.file_type == FileType.DOCUMENT
    assert info.provider_name == "local"
finally:
    os.unlink(temp_path)

# Test get_file_info with non-existent file
assert provider.get_file_info("/non/existent/path/file.xyz") is None

# Test scan with a temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    # Create some test files
    (Path(tmpdir) / "file1.py").write_text("print('hello')")
    (Path(tmpdir) / "file2.txt").write_text("hello")
    (Path(tmpdir) / "subdir").mkdir()
    (Path(tmpdir) / "subdir" / "file3.json").write_text("{}")
    
    # Scan all files
    config = ScanConfig(directories=[tmpdir], recursive=True)
    files = provider.scan([tmpdir], config)
    assert len(files) == 3
    
    # Scan with filter
    config_py = ScanConfig(
        directories=[tmpdir],
        recursive=True,
        filter_config=FilterConfig(extensions=['py'])
    )
    files = provider.scan([tmpdir], config_py)
    assert len(files) == 1
    assert files[0].extension == "py"
    
    # Scan non-recursive
    config_flat = ScanConfig(directories=[tmpdir], recursive=False)
    files = provider.scan([tmpdir], config_flat)
    assert len(files) == 2  # Only files in root, not subdir

print("LocalDiscoveryProvider tests passed!")

LocalDiscoveryProvider tests passed!


In [ ]:
from cjm_file_discovery.core.protocols import DiscoveryProvider

# Verify LocalDiscoveryProvider implements the protocol
assert isinstance(LocalDiscoveryProvider(), DiscoveryProvider)
print("Protocol implementation verified!")

Protocol implementation verified!


In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()